In [5]:
import time
import io
import math
import logging
import concurrent.futures
import numpy as np
import pdbfixer
import openmm
from openmm import app as openmm_app
from openmm import unit

In [6]:
class BatchForceFieldMinimizer:
    def __init__(self, stiffness=10.0, max_iterations=0, tolerance=2.39, platform_name='OpenCL', padding=5.0, cutoff=1.0, restore_local_center=False):
        """
        Args:
            stiffness: 限制力的强度 (kcal/mol/A^2)
            tolerance: 能量容差 (kcal/mol)
            platform_name: 计算平台 ('CUDA', 'OpenCL', 'CPU')
            padding: 体系间的安全间距 (nm), 必须不小于 2*cutoff
            cutoff: 非键相互作用截断 (nm)
            restore_local_center: 输出时是否加回每个体系的原始中心 (默认 False)
        """
        self.stiffness = stiffness
        self.max_iterations = max_iterations
        if isinstance(tolerance, unit.Quantity):
            self.tolerance = tolerance
        else:
            self.tolerance = tolerance * unit.kilocalories_per_mole
        self.platform_name = platform_name
        self.padding = padding * unit.nanometers
        self.cutoff = cutoff * unit.nanometers
        self.restore_local_center = restore_local_center
        if self.padding < 2 * self.cutoff:
            raise ValueError("Padding must be at least twice the cutoff distance to avoid interactions.")

        self.force_field = openmm_app.ForceField("charmm36.xml") 

    @staticmethod
    def _fix_single_pdb(pdb_str):
        fixer = pdbfixer.PDBFixer(pdbfile=io.StringIO(pdb_str))
        fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues()
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms(seed=0)
        fixer.addMissingHydrogens()
        handle = io.StringIO()
        openmm_app.PDBFile.writeFile(fixer.topology, fixer.positions, handle, keepIds=True)
        return handle.getvalue()

    @staticmethod
    def _get_num_atoms(topology):
        """Helper to count atoms regardless of OpenMM version."""
        if hasattr(topology, "getNumAtoms"):
            return topology.getNumAtoms()
        return sum(1 for _ in topology.atoms())

    def _center_by_ca(self, topology, positions):
        """
        Helper to center positions by CA atoms.
        Returns (centered_positions[nm], center_vec[nm])
        """
        pos_nm = positions.value_in_unit(unit.nanometers)
        ca_idx = [i for i, atom in enumerate(topology.atoms()) if getattr(atom, 'name', '') == 'CA']
        center = np.mean(pos_nm[ca_idx, :], axis=0)
        centered = pos_nm - center
        return unit.Quantity(centered, unit.nanometers), unit.Quantity(center, unit.nanometers)

    def run_batch(self, pdb_data_list, num_workers=None):
        """
        Batch Minimization of multiple PDB systems.
        Args:
            pdb_data_list: list include (id, pdb_string) tuples
            num_workers: number of parallel workers for fixing PDBs (0 or 1 for serial)
        Returns:
            result_map: {id: minimized_pdb_string}
        """
        total_systems = len(pdb_data_list)
        print(f"Starting batch processing for {total_systems} systems...")
        overall_start = time.time()
        stage_start = time.time()

        # pdb fixer
        fixed_systems = []

        if num_workers == 0 or num_workers == 1:
            print("num_workers <= 1, running fixer in serial mode.")
            for idx, (sys_id, pdb_str) in enumerate(pdb_data_list):
                fixed_pdb = self._fix_single_pdb(pdb_str)
                fixed_systems.append((idx, sys_id, fixed_pdb))
        else:
            with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers) as executor:
                future_to_meta = {
                    executor.submit(self._fix_single_pdb, pdb_str): (idx, sys_id)
                    for idx, (sys_id, pdb_str) in enumerate(pdb_data_list)
                }
                for future in concurrent.futures.as_completed(future_to_meta):
                    idx, sys_id = future_to_meta[future]
                    fixed_pdb = future.result()
                    fixed_systems.append((idx, sys_id, fixed_pdb))

        print("Structure fixing completed in {:.2f} seconds ({} valid systems).".format(time.time() - stage_start, len(fixed_systems)))
        
        # IMPORTANT! RECENTERING 
        stage_start = time.time()
        fixed_systems.sort(key=lambda x: x[0])
        ordered_systems = []
        center_map = {}  # {id: center_vec_nm}
        for _, sys_id, pdb_text in fixed_systems:
            pdb = openmm_app.PDBFile(io.StringIO(pdb_text))
            centered_pos, center_vec = self._center_by_ca(pdb.topology, pdb.positions)
            ordered_systems.append((sys_id, pdb.topology, centered_pos))
            center_map[sys_id] = center_vec

        # merging super system
        print("Merging systems into a single super-system...")
        first_id, first_top, first_pos = ordered_systems[0]
        modeller = openmm_app.Modeller(first_top, first_pos)

        # record atom index ranges for each subsystem for later splitting
        # format: {id: (start_atom_index, end_atom_index, shift_vector)}
        system_map = {}
        first_atom_count = self._get_num_atoms(first_top)
        system_map[first_id] = (0, first_atom_count, np.zeros(3, dtype=float))
        current_atom_count = first_atom_count

        # calculate grid size and positions
        n_systems = len(ordered_systems)
        grid_width = max(1, math.ceil(math.pow(n_systems, 1/3))) # simple cubic grid layout
        spacing_val = self.padding.value_in_unit(unit.nanometers)

        for i in range(1, n_systems):
            sys_id, top, pos = ordered_systems[i]
            ix = i % grid_width
            iy = (i // grid_width) % grid_width
            iz = i // (grid_width * grid_width)

            shift_vector = np.array([ix, iy, iz], dtype=float) * spacing_val
            pos_val = pos.value_in_unit(unit.nanometers)
            shifted_pos_val = pos_val + shift_vector
            shifted_pos = unit.Quantity(shifted_pos_val, unit.nanometers)

            modeller.add(top, shifted_pos)

            n_atoms = self._get_num_atoms(top)
            start_idx = current_atom_count
            end_idx = start_idx + n_atoms
            system_map[sys_id] = (start_idx, end_idx, shift_vector)
            current_atom_count = end_idx

        print("Merging completed in {:.2f} seconds.".format(time.time() - stage_start))
        print(f"Total merged atoms: {current_atom_count}")

        # 3. Create OpenMM System
        print(f"Creating simulation context for {current_atom_count} total atoms...")
        
        # Key: Use CutoffNonPeriodic to prevent interactions between distant systems
        # Without cutoff, computation scales as N^2 and is very slow
        stage_start = time.time()
        system = self.force_field.createSystem(
            modeller.topology,
            nonbondedMethod=openmm_app.CutoffNonPeriodic,
            nonbondedCutoff=self.cutoff,
            constraints=openmm_app.HBonds
        )
        print("System object created in {:.2f} seconds.".format(time.time() - stage_start))

        # 4. Add restraints
        # We need to add a CustomExternalForce
        force = openmm.CustomExternalForce("0.5 * k * ((x-x0)^2 + (y-y0)^2 + (z-z0)^2)")
        force.addGlobalParameter("k", self.stiffness)
        force.addPerParticleParameter("x0")
        force.addPerParticleParameter("y0")
        force.addPerParticleParameter("z0")
        
        positions_all = list(modeller.positions)
        for atom, pos in zip(modeller.topology.atoms(), positions_all):
            if atom.element.name != 'hydrogen':
                force.addParticle(atom.index, pos)
        
        system.addForce(force)

        # 5. Run minimization
        integrator = openmm.LangevinIntegrator(0,0.01,0.0)
        platform = openmm.Platform.getPlatformByName(self.platform_name)
        simulation = openmm_app.Simulation(modeller.topology, system, integrator, platform)
        simulation.context.setPositions(modeller.positions)
        
        print("Minimizing energy...")
        start_time = time.time()
        simulation.minimizeEnergy(maxIterations=self.max_iterations, tolerance=self.tolerance)
        end_time = time.time()
        print(f"Minimization complete in {end_time - start_time:.2f} seconds.")

        # 6. 拆分结果 (Demultiplexing)
        print("Splitting trajectories...")
        stage_start = time.time()
        state = simulation.context.getState(getPositions=True, getEnergy=True) # 获取总能量仅作参考
        all_positions = state.getPositions(asNumpy=True).value_in_unit(unit.angstroms)
        
        results = {}
        for sys_id, original_top, _ in ordered_systems:
            start, end, shift_vec_nm = system_map[sys_id]
            local_pos_angstroms = all_positions[start:end]
            shift_vec_angstroms = shift_vec_nm * 10.0
            final_pos = local_pos_angstroms - shift_vec_angstroms

            if self.restore_local_center:
                orig_center_ang = center_map[sys_id].value_in_unit(unit.nanometers) * 10.0
                final_pos = final_pos + orig_center_ang
            
            out_handle = io.StringIO()
            openmm_app.PDBFile.writeFile(
                original_top, 
                final_pos * unit.angstroms, 
                out_handle, 
                keepIds=True
            )
            
            # Add simple REMARKs to record layout offset and center vector (nm)
            pdb_text_out = out_handle.getvalue()
            shift_vals = shift_vec_nm
            center_vals = center_map[sys_id]
            remark1 = f"REMARK   1  SHIFT_VEC_NM   {shift_vals[0]:8.3f} {shift_vals[1]:8.3f} {shift_vals[2]:8.3f}\n"
            remark2 = f"REMARK   1  CENTER_VEC_NM  {center_vals[0]:8.3f} {center_vals[1]:8.3f} {center_vals[2]:8.3f}\n"
            lines = pdb_text_out.splitlines()
            lines.insert(0, remark2)
            lines.insert(0, remark1)
            results[sys_id] = "\n".join(lines) + "\n"
            
        print("Splitting completed in {:.2f} seconds.".format(time.time() - stage_start))
        print("Batch run finished in {:.2f} seconds.".format(time.time() - overall_start))
        return results

In [7]:
import os
import glob

pdb_files = glob.glob("example/*.pdb")
pdb_data_list = [(int(os.path.basename(f)[2]), f) for f in pdb_files]
batch_minimizer = BatchForceFieldMinimizer(platform_name='OpenCL')

In [8]:
batch_minimizer.run_batch(pdb_data_list, num_workers=None)

Starting batch processing for 9 systems...
Process pool failed (BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.). Falling back to threads.


Process SpawnProcess-10:
Traceback (most recent call last):
  File "/Users/megagatlingpea/miniforge3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/megagatlingpea/miniforge3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/megagatlingpea/miniforge3/lib/python3.12/concurrent/futures/process.py", line 252, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/megagatlingpea/miniforge3/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'BatchForceFieldMinimizer._fix_single_pdb' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>


IndexError: list index out of range

In [ ]:
# # bias logic
# import numpy as np
# all_shift_vec = []
# grid_width = 3.0
# for i in range(1, 27):

#     ix = i % grid_width
#     iy = (i // grid_width) % grid_width
#     iz = i // (grid_width * grid_width)

#     shift_vector = np.array([ix, iy, iz], dtype=float) * 5.0
#     all_shift_vec.append(shift_vector)
# all_shift_vec

[array([5., 0., 0.]),
 array([10.,  0.,  0.]),
 array([0., 5., 0.]),
 array([5., 5., 0.]),
 array([10.,  5.,  0.]),
 array([ 0., 10.,  0.]),
 array([ 5., 10.,  0.]),
 array([10., 10.,  0.]),
 array([0., 0., 5.]),
 array([5., 0., 5.]),
 array([10.,  0.,  5.]),
 array([0., 5., 5.]),
 array([5., 5., 5.]),
 array([10.,  5.,  5.]),
 array([ 0., 10.,  5.]),
 array([ 5., 10.,  5.]),
 array([10., 10.,  5.]),
 array([ 0.,  0., 10.]),
 array([ 5.,  0., 10.]),
 array([10.,  0., 10.]),
 array([ 0.,  5., 10.]),
 array([ 5.,  5., 10.]),
 array([10.,  5., 10.]),
 array([ 0., 10., 10.]),
 array([ 5., 10., 10.]),
 array([10., 10., 10.])]